In [ ]:
pip install pyspark

In [ ]:
import os
import sys
import pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, when
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# 1. Clean up any stuck sessions
SparkContext._active_spark_context = None
if 'py4j' in sys.modules:
    try: spark.stop()
    except: pass

# 2. Environment Variables setup
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ.get("PATH", "")

# 3. Dynamic version resolution
spark_version = pyspark.__version__
major_version = int(spark_version.split(".")[0])
scala_version = "2.13" if major_version >= 4 else "2.12"

print(f"Initializing Spark {spark_version} with Scala {scala_version}...")

# 4. Initialize Spark Session (Added PostgreSQL Driver here)
spark = SparkSession.builder \
    .appName("GreenInnovationWeatherProcessor") \
    .config("spark.jars.packages", f"org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version},org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Define the data schema
schema = StructType([
    StructField("city", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("temp", DoubleType(), True),
    StructField("humidity", IntegerType(), True),
    StructField("pressure", IntegerType(), True),
    StructField("sea_level", IntegerType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("clouds", IntegerType(), True),
    StructField("rain_1h", DoubleType(), True),
    StructField("weather_description", StringType(), True),
    StructField("timestamp", DoubleType(), True)
])

# 5. Connect to Kafka and start the stream
print("Connecting to Kafka and starting data stream...")
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "weather_data") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .option("kafka.group.id", "weather_processor_final_v2") \
    .load()

# Parse JSON payload
weather_df = kafka_df.selectExpr("CAST(value AS STRING) as json_value") \
    .select(from_json(col("json_value"), schema).alias("data")) \
    .select("data.*")

# Data Processing: Add weather classification logic
processed_df = weather_df.withColumn(
    "weather_status",
    when(col("temp") > 30, "Hot")
    .when(col("temp") >= 15, "Moderate")
    .otherwise("Cold")
)

# 6. PostgreSQL Connection Properties
jdbc_url = "jdbc:postgresql://localhost:5432/green_innovation_db"
db_properties = {
    "user": "postgres",
    "password": "admin",
    "driver": "org.postgresql.Driver"
}

# Function to write each micro-batch to PostgreSQL
def write_to_postgres(batch_df, batch_id):
    batch_df.write \
        .jdbc(
            url=jdbc_url,
            table="weather_data",
            mode="append",
            properties=db_properties
        )

print("Starting to write stream to PostgreSQL...")

# 7. Start streaming using foreachBatch
query = processed_df.writeStream \
    .foreachBatch(write_to_postgres) \
    .start()

query.awaitTermination()

Initializing Spark 4.0.3 with Scala 2.13...
Connecting to Kafka and starting data stream...
Streaming is running in the background! Run the next cell to see the data...


In [2]:
import time
from IPython.display import clear_output

# الكود ده هيعمل تحديث للشاشة كل 5 ثواني ويطبع الداتا اللي السبارك بيعالجها
try:
    print("Fetching data from memory...")
    for i in range(20):  # هيعمل تحديث 20 مرة
        clear_output(wait=True)
        print(f"--- Data Update {i+1} ---")
        
        # بنقرأ الجدول الوهمي اللي السبارك بيكتب فيه
        spark.sql("SELECT * FROM weather_table").show(truncate=False)
        
        time.sleep(5)
except KeyboardInterrupt:
    print("Display stopped by user.")

--- Data Update 20 ---
+-----+--------+---------+-----+--------+--------+---------+----------+------+-------+-------------------+--------------------+--------------+
|city |latitude|longitude|temp |humidity|pressure|sea_level|wind_speed|clouds|rain_1h|weather_description|timestamp           |weather_status|
+-----+--------+---------+-----+--------+--------+---------+----------+------+-------+-------------------+--------------------+--------------+
|Minya|28.0871 |30.7618  |35.87|14      |1009    |1009     |5.78      |0     |0.0    |clear sky          |1.7835116952844803E9|Hot           |
|Minya|28.0871 |30.7618  |35.87|14      |1009    |1009     |5.78      |0     |0.0    |clear sky          |1.7835117012381587E9|Hot           |
|Minya|28.0871 |30.7618  |35.87|14      |1009    |1009     |5.78      |0     |0.0    |clear sky          |1.7835120108009658E9|Hot           |
|Minya|28.0871 |30.7618  |35.87|14      |1009    |1009     |5.78      |0     |0.0    |clear sky          |1.78351207665